# Final Project: Statistics for Data Science with Python
Boston Housing Dataset Analysis

In this project, we will play the role of a Data Scientist at a housing agency. We will analyze the Boston Housing dataset to extract meaningful insights using descriptive statistics, visualizations, and hypothesis testing.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

In [ ]:
boston_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv'
boston_df = pd.read_csv(boston_url)

boston_df.head()

## Task 1: Descriptive Statistics and Visualizations
For all visualizations, we will use Seaborn and Matplotlib to explore the dataset variables.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(y='MEDV', data=boston_df, color='skyblue')
plt.title('Boxplot of Median Value of Owner-Occupied Homes (MEDV)')
plt.ylabel('Median Value ($1000s)')
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.countplot(x='CHAS', data=boston_df, palette='Set2')
plt.title('Bar Plot of Charles River Tract Bound')
plt.xlabel('Bound by Charles River (0 = No, 1 = Yes)')
plt.ylabel('Count of Tracts')
plt.show()

In [ ]:
boston_df.loc[(boston_df['AGE'] <= 35), 'age_group'] = '35 years and younger'
boston_df.loc[(boston_df['AGE'] > 35) & (boston_df['AGE'] < 70), 'age_group'] = 'Between 35 and 70 years'
boston_df.loc[(boston_df['AGE'] >= 70), 'age_group'] = '70 years and older'

plt.figure(figsize=(10, 6))
sns.boxplot(x='age_group', y='MEDV', data=boston_df, order=['35 years and younger', 'Between 35 and 70 years', '70 years and older'], palette='pastel')
plt.title('Median Home Values by Age of Structure')
plt.xlabel('Age Group (Proportion of owner-occupied units built prior to 1940)')
plt.ylabel('Median Value ($1000s)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='INDUS', y='NOX', data=boston_df, alpha=0.6, color='darkred')
plt.title('Nitric Oxide Concentrations vs. Non-Retail Business Acres')
plt.xlabel('Proportion of Non-Retail Business Acres per Town (INDUS)')
plt.ylabel('Nitric Oxides Concentration (NOX, parts per 10 million)')
plt.show()

Conclusion for Scatter Plot: 
There appears to be a positive, linear relationship between the proportion of non-retail business acres (INDUS) and nitric oxide concentrations (NOX). As industrial acres increase, nitric oxide concentrations also tend to increase.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(boston_df['PTRATIO'], bins=15, kde=True, color='purple')
plt.title('Histogram of Pupil-Teacher Ratio by Town')
plt.xlabel('Pupil-Teacher Ratio (PTRATIO)')
plt.ylabel('Frequency')
plt.show()

---
## Task 2: Hypothesis Testing
We will use a significance level of alpha = 0.05 for all hypothesis tests.

### 1. T-test: Is there a significant difference in median value of houses bounded by the Charles river or not?
*   Null Hypothesis (H0): There is no significant difference in median values of houses bounded by the Charles river and those that are not.
*   Alternative Hypothesis (H1): There is a significant difference in median values of houses bounded by the Charles river and those that are not.

In [ ]:
stats.ttest_ind(boston_df[boston_df['CHAS'] == 0]['MEDV'],
                boston_df[boston_df['CHAS'] == 1]['MEDV'], equal_var=True)

Conclusion: 
Since the p-value (less than 0.05) is smaller than our alpha level, we reject the null hypothesis. There is sufficient evidence to conclude that there is a statistically significant difference in the median values of houses based on whether they are bounded by the Charles River.

### 2. ANOVA: Is there a difference in Median values of houses (MEDV) for each proportion of owner-occupied units built prior to 1940 (AGE)?
*   Null Hypothesis (H0): The means of MEDV are the same across all three age groups.
*   Alternative Hypothesis (H1): At least one of the age groups has a different mean MEDV.

In [ ]:
low_age = boston_df[boston_df['age_group'] == '35 years and younger']['MEDV']
mid_age = boston_df[boston_df['age_group'] == 'Between 35 and 70 years']['MEDV']
high_age = boston_df[boston_df['age_group'] == '70 years and older']['MEDV']

stats.f_oneway(low_age, mid_age, high_age)

Conclusion: 
Since the p-value is extremely small (less than 0.05), we reject the null hypothesis. There is strong evidence to suggest that there is a significant difference in median house values across the different age groups of the properties.

### 3. Pearson Correlation: Can we conclude that there is no relationship between Nitric oxide concentrations and proportion of non-retail business acres per town?
*   Null Hypothesis (H0): There is no correlation between Nitric oxide concentrations (NOX) and proportion of non-retail business acres (INDUS).
*   Alternative Hypothesis (H1): There is a correlation between Nitric oxide concentrations (NOX) and proportion of non-retail business acres (INDUS).

In [ ]:
stats.pearsonr(boston_df['NOX'], boston_df['INDUS'])

Conclusion: 
Since the p-value is less than 0.05, we reject the null hypothesis. The high positive correlation coefficient (~0.76) indicates a strong positive linear relationship between nitric oxide concentrations and the proportion of non-retail business acres per town.

### 4. Regression Analysis: What is the impact of an additional weighted distance to the five Boston employment centres on the median value of owner-occupied homes?
*   Null Hypothesis (H0): Weighted distances to five Boston employment centers (DIS) has no effect on the median value of owner-occupied homes (MEDV).
*   Alternative Hypothesis (H1): Weighted distances to five Boston employment centers (DIS) has an effect on the median value of owner-occupied homes (MEDV).

In [ ]:
X = boston_df['DIS']
y = boston_df['MEDV']

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

model.summary()

Conclusion: 
The p-value for the DIS coefficient is less than 0.05, so we reject the null hypothesis. The coefficient for DIS is 1.0916, which means that for every additional unit increase in the weighted distance to the employment centers, the median value of owner-occupied homes increases by approximately $1,091.60.